In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/NLP_final_project/news_relevant_notebook2.parquet"
df = pd.read_parquet(file_path, engine="pyarrow")

print(df.shape)
print(df.columns.tolist())
df.head()

(129326, 19)
['url', 'date', 'language', 'title', 'text', 'full_text', 'title_len', 'text_len', 'full_text_len', 'text_for_match', 'ai_core_hits', 'impact_hits', 'business_hits', 'industry_hits', 'relevance_score', 'has_ai_signal', 'is_relevant_rule', 'exclude_hits', 'is_relevant']


,url,date,language,title,text,full_text,title_len,text_len,full_text_len,text_for_match,ai_core_hits,impact_hits,business_hits,industry_hits,relevance_score,has_ai_signal,is_relevant_rule,exclude_hits,is_relevant
0,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,78,4820,4899,this ai video of gymnastics might be the freak...,1,0,1,0,4,True,True,0,True
1,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...",54,5123,5178,if using ai feels like a chore try this boing ...,6,0,0,1,19,True,True,0,True
2,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,en,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,102,4017,4120,the road ahead how china s ai foundation model...,4,3,0,1,19,True,True,0,True
3,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,en,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,65,4290,4356,microsoft and nvidia to empower developers wit...,3,1,0,0,11,True,True,0,True
4,https://citylife.capetown/lb/uncategorized/how...,2023-12-12,en,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong C...,75,2579,2655,google releases new chatbot bard as a strong c...,6,0,0,0,18,True,True,0,True


In [3]:
keep_cols = [col for col in [
    "url", "date", "title", "text", "full_text",
    "relevance_score", "ai_core_hits", "impact_hits",
    "business_hits", "industry_hits"
] if col in df.columns]

df = df[keep_cols].copy()
print(df.shape)

(129326, 10)


In [4]:
INDUSTRY_DICT = {
    "Healthcare": [
        "healthcare", "hospital", "medical", "medicine", "clinical",
        "patient", "radiology", "diagnostic", "pharma", "pharmaceutical",
        "biotech", "drug discovery"
    ],
    "Finance": [
        "finance", "financial", "bank", "banking", "insurance",
        "fintech", "asset management", "investment", "trading", "lender"
    ],
    "Legal": [
        "legal", "law firm", "lawyer", "attorney", "contract review",
        "litigation", "compliance"
    ],
    "Manufacturing": [
        "manufacturing", "factory", "industrial", "supply chain",
        "production", "assembly", "plant operations"
    ],
    "Retail & E-commerce": [
        "retail", "e commerce", "ecommerce", "merchant", "consumer goods",
        "shopping", "online marketplace"
    ],
    "Education": [
        "education", "school", "student", "teacher", "classroom",
        "university", "college", "learning platform"
    ],
    "Media & Marketing": [
        "media", "advertising", "marketing", "publisher", "newsroom",
        "content creation", "ad tech"
    ],
    "Technology": [
        "software", "cloud", "developer", "saas", "semiconductor",
        "chip", "data center", "platform", "tech company"
    ],
    "Transportation & Logistics": [
        "logistics", "transportation", "shipping", "delivery",
        "warehouse", "fleet", "automotive", "autonomous vehicle"
    ],
    "Government & Public Sector": [
        "government", "public sector", "federal", "agency",
        "municipal", "defense", "military"
    ],
    "Energy & Utilities": [
        "energy", "utility", "power grid", "electricity", "oil",
        "gas", "renewable", "solar", "wind"
    ],
    "Telecom": [
        "telecom", "telecommunications", "wireless", "broadband",
        "network operator"
    ],
    "Consulting & Professional Services": [
        "consulting", "consultant", "professional services",
        "advisory", "outsourcing"
    ]
}

In [7]:
import re
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["industry_text"] = df["full_text"].apply(normalize_text)

In [9]:
def get_matching_industries(text, industry_dict):
    matches = []
    for industry, keywords in industry_dict.items():
        for kw in keywords:
            pattern = r"\b" + re.escape(kw.lower()) + r"\b"
            if re.search(pattern, text):
                matches.append(industry)
                break
    return matches

df["industries"] = df["industry_text"].apply(lambda x: get_matching_industries(x, INDUSTRY_DICT))
df["industry_count"] = df["industries"].apply(len)

df[["title", "industries", "industry_count"]].head(10)

,title,industries,industry_count
0,This AI video of gymnastics might be the freak...,[Media & Marketing],1
1,"If using AI feels like a chore, try this - Boi...","[Media & Marketing, Technology]",2
2,The Road Ahead: How China's AI Foundation Mode...,"[Technology, Transportation & Logistics]",2
3,Microsoft and Nvidia to Empower Developers wit...,[Technology],1
4,Google Releases New Chatbot Bard as a Strong C...,[],0
5,Zoom Expands AI Offering with AI Companion and...,"[Technology, Transportation & Logistics, Telecom]",3
6,Pro-AI Thinking: Enhancing Industrial Environm...,"[Healthcare, Finance, Legal, Manufacturing, Te...",7
7,Best AI Prompts for Business Risk Management,"[Finance, Legal, Manufacturing]",3
8,Bullfrog AI Holdings Inc. [BFRG] Revenue clock...,[Finance],1
9,IIM Ahmedabad student writes project using Cha...,"[Finance, Manufacturing, Retail & E-commerce, ...",8


In [10]:
df["industry_count"].value_counts().sort_index()

,count
industry_count,
0,7046
1,9847
2,16244
3,21593
4,22076
5,17731
6,12601
7,8124
8,4969


In [11]:
print("No-industry articles:", (df["industry_count"] == 0).sum())
print("Pct no-industry:", round((df["industry_count"] == 0).mean() * 100, 2), "%")

No-industry articles: 7046
Pct no-industry: 5.45 %


In [12]:
df_industries = df.explode("industries").copy()
df_industries = df_industries[df_industries["industries"].notna()]

industry_summary = (
    df_industries.groupby("industries")
    .size()
    .reset_index(name="article_count")
    .sort_values("article_count", ascending=False)
)

industry_summary.head(20)

,industries,article_count
8,Media & Marketing,89912
10,Technology,78366
3,Finance,58027
1,Education,57106
4,Government & Public Sector,46673
5,Healthcare,36964
2,Energy & Utilities,35391
7,Manufacturing,34599
6,Legal,33860
12,Transportation & Logistics,26247


In [13]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 112.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [14]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [17]:
import spacy
from tqdm.auto import tqdm

nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "lemmatizer", "attribute_ruler"])

texts = (
    df["title"].fillna("").astype(str) + " " +
    df["text"].fillna("").astype(str).str[:1000]
).tolist()

org_entities_raw = []

for doc in tqdm(nlp.pipe(texts, batch_size=64), total=len(texts)):
    orgs = [ent.text.strip() for ent in doc.ents if ent.label_ == "ORG"]
    org_entities_raw.append(orgs)

df["org_entities_raw"] = org_entities_raw

  0%|          | 0/129326 [00:00<?, ?it/s]

In [18]:
BAD_ORGS = {
    "AI", "A.I.", "ML", "GPT", "ChatGPT", "OpenAI API",
    "Generative AI", "GenAI"
}

DROP_ORGS = {
    "Reuters", "Bloomberg", "The Associated Press", "AP",
    "United States", "U.S.", "US", "Congress", "Senate",
    "White House", "European Union", "EU"
}

def clean_org_list(orgs):
    cleaned = []
    for org in orgs:
        org = str(org).strip()
        org = re.sub(r"\s+", " ", org)
        if len(org) < 2:
            continue
        if org in BAD_ORGS or org in DROP_ORGS:
            continue
        if org.lower() in {"ai", "ml", "gpt", "chatgpt"}:
            continue
        cleaned.append(org)
    return list(dict.fromkeys(cleaned))

df["org_entities"] = df["org_entities_raw"].apply(clean_org_list)
df["org_count"] = df["org_entities"].apply(len)

df[["title", "org_entities", "org_count"]].sample(10, random_state=42)

,title,org_entities,org_count
89071,Alida Launches Industry-First AI Assistant for...,[ResearchContact UsContact UsNewsroomServices ...,5
123265,Suhel Seth on Future Impact of Artificial Inte...,[Advertising & Branding – Brown Pundits Suhel ...,6
62387,ASTERRA's new satellite based PolSAR technolog...,"[ASTERRA, PolSAR technology, the DanubeWeather...",8
4270,Where federal AI is headed next | Federal News...,"[Federal News Network Where, Federal News Netw...",4
56300,Exotel launches AI chatbot builder ‘ExoMind’ p...,"[Telecom News, ET Telecom, Google]",3
27153,Subtle Medical Joins Imaging AI in Practice De...,"[Subtle Medical Joins Imaging AI, Practice Dem...",7
22358,Wadhwani AI showcases Solutions for Social Goo...,[Wadhwani AI showcases Solutions for Social Go...,9
76152,Zoom and CrowdStrike stocks ride SaaS’s AI tre...,"[CrowdStrike, CMC Markets Zoom, CMC Markets, O...",6
15663,Venzee AI Accelerates Intelligent Data Syndica...,[Venzee AI Accelerates Intelligent Data Syndic...,4
10415,This AI will tell you whether you're being pai...,"[NewsBreak, inHomeLocalHeadlinesCoronavirusOri...",5


In [19]:
df_orgs = df.explode("org_entities").copy()
df_orgs = df_orgs[df_orgs["org_entities"].notna()]

company_summary = (
    df_orgs.groupby("org_entities")
    .size()
    .reset_index(name="mention_count")
    .sort_values("mention_count", ascending=False)
)

company_summary.head(30)

,org_entities,mention_count
92587,Microsoft,5219
63446,Google,5015
105669,OpenAI,4227
118718,Rawpixel Ltd.,4123
150285,U.S. State,3478
102508,Newswires U.S. & International,3477
74939,Industry Newswires,3477
150286,U.S. State Archive,3477
88980,Major News,3462
130581,Sites U.S. TV & Radio Stations U.S. & Internat...,3462


In [20]:
df_link = df.copy()
df_link = df_link.explode("industries")
df_link = df_link.explode("org_entities")
df_link = df_link[
    df_link["industries"].notna() &
    df_link["org_entities"].notna()
].copy()

industry_company_summary = (
    df_link.groupby(["industries", "org_entities"])
    .size()
    .reset_index(name="co_mentions")
    .sort_values(["industries", "co_mentions"], ascending=[True, False])
)

industry_company_summary.head(30)

,industries,org_entities,co_mentions
12903,Consulting & Professional Services,Industry Newswires,713
17872,Consulting & Professional Services,Newswires U.S. & International,713
25756,Consulting & Professional Services,U.S. State,713
25757,Consulting & Professional Services,U.S. State Archive,713
15490,Consulting & Professional Services,Major News,709
22330,Consulting & Professional Services,Sites U.S. TV & Radio Stations U.S. & Internat...,709
29056,Consulting & Professional Services,the News Pricing Distribution Distribution Ove...,524
6046,Consulting & Professional Services,Commodoties Oil & Energy Economic Calender Res...,523
11949,Consulting & Professional Services,Home News News,523
12896,Consulting & Professional Services,Industry News,523


In [21]:
top_companies_by_industry = (
    industry_company_summary
    .groupby("industries", group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

top_companies_by_industry.head(50)

,industries,org_entities,co_mentions
0,Consulting & Professional Services,Industry Newswires,713
1,Consulting & Professional Services,Newswires U.S. & International,713
2,Consulting & Professional Services,U.S. State,713
3,Consulting & Professional Services,U.S. State Archive,713
4,Consulting & Professional Services,Major News,709
5,Consulting & Professional Services,Sites U.S. TV & Radio Stations U.S. & Internat...,709
6,Consulting & Professional Services,the News Pricing Distribution Distribution Ove...,524
7,Consulting & Professional Services,Commodoties Oil & Energy Economic Calender Res...,523
8,Consulting & Professional Services,Home News News,523
9,Consulting & Professional Services,Industry News,523


In [22]:
df.to_parquet(
    "/content/drive/MyDrive/NLP_final_project/news_entities_notebook3.parquet",
    engine="pyarrow",
    index=False
)

company_summary.to_csv(
    "/content/drive/MyDrive/NLP_final_project/company_summary_notebook3.csv",
    index=False
)

industry_company_summary.to_csv(
    "/content/drive/MyDrive/NLP_final_project/industry_company_summary_notebook3.csv",
    index=False
)

top_companies_by_industry.to_csv(
    "/content/drive/MyDrive/NLP_final_project/top_companies_by_industry_notebook3.csv",
    index=False
)

print("saved Notebook 3 outputs")

saved Notebook 3 outputs
